# Task 1.2 — Key Assumptions
**Paper:** One-Sided Support Vector Regression for Multiclass Cost-Sensitive Classification  
**Authors:** Han-Hsing Tu, Hsuan-Tien Lin  
**Venue:** ICML 2010  
**Student:** Vaishnav Verma (230160)

## Assumption 1: The cost matrix is known, fixed, and defined before training

**Assumption:** OSSVR requires a fully specified K×K cost matrix C where every entry C[y, k] (cost of predicting class k when true class is y) is known before training begins. The entire regression target — the regret vector — is derived from this matrix.

**Why the method needs it:** The regret vector rₙ[k] = C[yₙ, k] − C[yₙ, yₙ] is computed directly from the cost matrix. Without a fixed C, there is no regression target, and the K one-sided SVR models cannot be trained. The cost matrix is not learned — it is an input to the algorithm.

**Violation scenario:** In medical diagnosis, the cost of misclassifying a patient may depend on the patient's individual risk profile (age, comorbidities), making costs example-dependent rather than class-level. OSSVR's fixed cost matrix cannot capture such instance-varying costs.

**Paper reference:** Section 2 (Problem Setup), where C is defined as a fixed K×K matrix; Section 3, Equations (3)–(4), where regret vectors are derived from C.

## Assumption 2: The regret function for each class can be well-approximated by a smooth function in the kernel-induced feature space

**Assumption:** OSSVR assumes that for each class k, there exists a smooth function fₖ(x) = wₖ · Φ(x) + bₖ in the reproducing kernel Hilbert space (RKHS) induced by the chosen kernel (e.g., RBF) that can approximate the regret values across the input space. In other words, nearby points in kernel space should have similar regret values.

**Why the method needs it:** The one-sided SVR regressor is a kernel machine — it represents fₖ as a linear combination of kernel evaluations. If the mapping from input space to regret is highly discontinuous or if the chosen kernel cannot capture the decision boundaries between cost regions, the regressor will produce poor approximations and the argmin prediction will be unreliable.

**Violation scenario:** A dataset where two very similar inputs (close in all features) belong to different classes with vastly different cost profiles. For example, in fraud detection, two nearly identical transactions may have very different costs of misclassification if one is a genuine high-value transaction and the other is fraud. The regret surface would be discontinuous, and a smooth kernel regressor would struggle.

**Paper reference:** Section 3.1 (Dual Formulation), where fₖ(x) is parameterised as a kernel expansion; the implicit smoothness assumption comes from the RKHS regularisation ½‖wₖ‖² in Equation (5).

## Assumption 3: Independent one-versus-all decomposition is sufficient to capture inter-class cost relationships

**Assumption:** OSSVR trains K independent regressors — one per class — and combines their outputs via argmin at prediction time. It assumes that the optimal cost-sensitive decision boundary can be reconstructed from K independently trained regression functions, without modelling pairwise or higher-order interactions between classes.

**Why the method needs it:** The OVA decomposition means each regressor fₖ is trained without any knowledge of what the other K−1 regressors are learning. The argmin prediction rule assumes that the independently estimated regret values are calibrated well enough that their relative ordering correctly identifies the cheapest class. If the regressors are miscalibrated relative to each other (e.g., one regressor systematically over-estimates while another under-estimates), the argmin will be incorrect even if each regressor individually fits well.

**Violation scenario:** In a 5-class problem where the cost of confusing class A with class B is orders of magnitude different from confusing A with C, the regressor for class A may need to be aware of the specific alternative being considered. For instance, in a multi-disease diagnosis where treating disease A as disease B is fatal but treating A as C is merely suboptimal, independent regressors may not properly calibrate the relative scale of their outputs.

**Paper reference:** Section 3 (Method), where the K regressors are trained independently; the argmin prediction rule assumes comparability across independently trained fₖ outputs. The paper acknowledges this is an OVA approach but does not discuss calibration across regressors.